# Load deps

In [ ]:
# !pip install -q torcheval

In [ ]:
# # if src modules imported
# from google.colab import drive
# drive.mount('/content/drive')
# import sys
# app_path = '/content/drive/MyDrive/Projects/miniSD'
# sys.path.append(app_path)

In [ ]:
import torch,timm,torchvision
import torch.nn.functional as F
from torchvision import transforms
from torch import tensor,nn
from torch.utils.data import DataLoader

from src.utils import store_attr
from src.conv import def_device

from src.datasets import show_image, show_images, DataLoaders
from src.learner import Learner, TrainCB, ProgressCB, MetricsCB, DeviceCB, Callback, to_cpu

In [ ]:
#|export --> utils.net
import urllib, json
from urllib.error import HTTPError
from urllib.parse import urlencode, urlparse, urlunparse
from urllib.request import Request


url_default_headers = {
    "Accept":
    "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.9",
    "Accept-Language": "en-US,en;q=0.9",
    "Cache-Control": "max-age=0",
    "Sec-Fetch-Dest": "document",
    "Sec-Fetch-Mode": "navigate",
    "Sec-Fetch-Site": "none",
    "Sec-Fetch-User": "?1",
    "Upgrade-Insecure-Requests": "1",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/87.0.4280.88 Safari/537.36"
}

class _SafeRedirectHandler(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        if code in (307, 308):
            new_req = Request(newurl, data=req.data, method=req.get_method())
            for k,v in req.headers.items(): new_req.add_header(k, v)
        else: new_req = super().redirect_request(req, fp, code, msg, headers, newurl)
        if new_req and urlparse(newurl).netloc != urlparse(req.full_url).netloc: new_req.remove_header('Authorization')
        return new_req

def urlopener():
    _opener = urllib.request.build_opener(_SafeRedirectHandler)
    _opener.addheaders = list(url_default_headers.items())
    return _opener


def urlquote(url):
    "Update url's path with `urllib.parse.quote`"
    subdelims = "!$&'()*+,;="
    gendelims = ":?#[]@"
    safe = subdelims+gendelims+"%/"
    p = list(urlparse(url))
    p[2] = urllib.parse.quote(p[2], safe=safe)
    for i in range(3,6): p[i] = urllib.parse.quote(p[i], safe=safe)
    return urlunparse(p)

def urlwrap(url, data=None, headers=None):
    "Wrap `url` in a urllib `Request` with `urlquote`"
    return url if isinstance(url,Request) else Request(urlquote(url), data=data, headers=headers or {})

def urlopen(url, data=None, headers=None, timeout=None, **kwargs):
    "Like `urllib.request.urlopen`, but first `urlwrap` the `url`, and encode `data`"
    if kwargs and not data: data=kwargs
    if data is not None:
        if not isinstance(data, (str,bytes)): data = urlencode(data)
        if not isinstance(data, bytes): data = data.encode('ascii')
    try: return urlopener().open(urlwrap(url, data=data, headers=headers), timeout=timeout)
    except HTTPError as e: 
        e.msg += f"\n====Error Body====\n{e.read().decode(errors='ignore')}"
        raise


def urlread(url, data=None, headers=None, decode=True, return_json=False, return_headers=False, timeout=None, **kwargs):
    "Retrieve `url`, using `data` dict or `kwargs` to `POST` if present"
    try:
        with urlopen(url, data=data, headers=headers, timeout=timeout, **kwargs) as u: res,hdrs = u.read(),u.headers
    except HTTPError as e:
        e.msg += f"\n====Error Body====\n{e.read().decode(errors='ignore')}"
        raise

    if decode: res = res.decode()
    if return_json: res = json.loads(res)
    return (res,dict(hdrs)) if return_headers else res

In [ ]:
face_url = "https://images.pexels.com/photos/2690323/pexels-photo-2690323.jpeg?w=256"
spiderweb_url = "https://images.pexels.com/photos/34225/spider-web-with-water-beads-network-dewdrop.jpg?w=256"

# Load and preview Images

In [ ]:
def download_image(url):
    imgb = urlread(url, decode=False)
    return torchvision.io\
        .decode_image(tensor(list(imgb), dtype=torch.uint8))\
        .float()/255.

content_im = download_image(face_url).to(def_device)
print('content_im.shape:', content_im.shape)
show_image(content_im);
print(f"image bounds: min = {content_im.min()} | max = {content_im.max()}") # Check bounds

# Optimizing Images

- We're using dummy Model and Dataset objects in order to use our current training tools
- But optimizing an image instead of a model

In [ ]:
class LengthDataset():
    def __init__(self, length=1): self.length=length
    def __len__(self): return self.length
    def __getitem__(self, idx): return 0,0

def get_dummy_dls(length=100):
    return DataLoaders(
        DataLoader(LengthDataset(length), batch_size=1), # Train
        DataLoader(LengthDataset(1), batch_size=1)       # Valid (length 1)
    )

class TensorModel(nn.Module):
    def __init__(self, t):
        super().__init__()
        self.t = nn.Parameter(t.clone())
    def forward(self, x=0): return self.t

class ImageOptCB(TrainCB):
    def predict(self, learn): learn.preds = learn.model()
    def get_loss(self, learn): learn.loss = learn.loss_func(learn.preds)

def loss_fn_mse(im):
    return F.mse_loss(im, content_im)

In [ ]:
model = TensorModel(torch.rand_like(content_im))
print([p.shape for p in model.parameters()])
print("====================")
show_image(model());

In [ ]:
model = TensorModel(torch.rand_like(content_im))
cbs = [ImageOptCB(), ProgressCB(), MetricsCB(), DeviceCB()]
learn = Learner(
    model,
    get_dummy_dls(100),
    loss_fn_mse,
    lr=1e-2,
    cbs=cbs,
    opt_func=torch.optim.Adam
)
learn.fit(1)

# Result (left) vs target image (right):
show_images([learn.model().clip(0, 1), content_im]);

## Viewing optimization progress

- It would be great if we could see what is happening over time.
- We could save individual images and turn them into a video
- but for quick feedback we can also log images every few iterations and display them in a grid in `after_fit`

In [ ]:
class ImageLogCB(Callback):
    order = ProgressCB.order + 1
    def __init__(self, log_every=10):
        store_attr()
        self.images=[]
        self.i=0
    def after_batch(self, learn): 
        if self.i%self.log_every == 0:
            self.images.append(to_cpu(learn.preds.clip(0, 1)))
        self.i += 1
    def after_fit(self, learn): show_images(self.images)

In [ ]:
model = TensorModel(torch.rand_like(content_im))
learn = Learner(model, get_dummy_dls(150), loss_fn_mse, 
                lr=1e-2, cbs=cbs, opt_func=torch.optim.Adam)
learn.fit(1, cbs=[ImageLogCB(30)])

In [ ]:
# L20 | 34:57